In [ ]:
from numpy.random.mtrand import f
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Callable, Iterator, Dict
from concurrent.futures import ProcessPoolExecutor
from scipy.special import lambertw
import multiprocessing
import pandas as pd
import sys
from sklearn.metrics import r2_score, mean_absolute_error
import warnings


def is_notebook():
    return "ipykernel" in sys.modules


# ANALYSIS FOR THE TWO ALLELE, TIME-INVARIANT FITNESS CASE

# global random number generator
rng = np.random.default_rng()


# wrightfishersampling simulates natural selection based on initial allele counts and a list of functions describing
# the fitness of each allele at each generation in time
def wrightfishersampling(K: np.ndarray, Ff: List[Callable[[int], float]]) -> Iterator[np.ndarray]:
    """
    params
    K = numpy array of initial allele count for each allele. for now, only 2 alleles
    Ff = Fitness function. change in F. list of lambda functions that change F
           based on the generation number for each allele.
           for now, fitness functions are constant (eg always return 1)
          input should be just the generation number (irrelevent here).
          output is fitness.

    yields np.ndarray: allele counts for current gen
    """

    # population number no longer needs to be an input because it is now implied
    N = np.sum(K)

    # input validation

    if np.any(K<=0):
      raise ValueError("Number of alleles cannot be negative")
    if N == 0:
        raise ValueError("error: population number is zero")
    if not isinstance(Ff, list) or not all(callable(x) for x in Ff):
        raise ValueError("Ff must be a list of callable functions")
    if not (K.size == len(Ff)):
        raise ValueError("K and Ff must have the same length")
    if K.size < 1:
        raise ValueError("K must be at least 1")

    t = 0  # we assume that the first generation is 0

    # while none of the alleles have yet fixed
    while True:

        yield K

        if np.max(K) == N:
            break

        # p is unnormalized initially because the equations in Ff model the fitness of each allele at each time
        # in relative terms rather than proportions to make it much easier to input
        F_values_at_t = np.array([func(t) for func in Ff],  dtype=float)
        p_unnormalized = K * F_values_at_t

        sum_p_unnormalized = np.sum(p_unnormalized)

        if sum_p_unnormalized == 0 or np.isnan(sum_p_unnormalized):
            pvals = np.zeros_like(K, dtype=float)
        else:
            # normalize
            pvals = p_unnormalized / sum_p_unnormalized
            # force the sum of pvals to be exactly 1.0 to prevent base 2 rounding errors
            pvals /= np.sum(pvals)

        # random multinomial is implemented to select the next generation using the pvals as likelihoods of
        # each allele.

        new_counts = rng.multinomial(n=N, pvals=pvals)
        K = new_counts

        t += 1  # next generation begins and the loop restarts

def run_single_simulation(initial_K_counts: np.ndarray, Ff: List[Callable[[int], float]]) -> Dict:
   """
   params
     initial_K_counts : np.ndarray, array representing the initial allele counts
     Ff : List[Callable[[int], float]], list of fitness functions for each allele

   returns
     dict containing:
       "fixed" bool indicating if an allele fixed
       "final_generations" total num of gens simulated
       "allele_counts_history" np.ndarray w/ allele counts per gen
       "allele_frequencies_history" np.ndarray w/ allele frequencies per gen
   """
   # get simulation history using vstack
   history = np.vstack(list(wrightfishersampling(initial_K_counts, Ff)))
   N = int(initial_K_counts.sum())

   final_t = history.shape[0]
   fixed = (history[-1] == N).any()
   # check if the first allele fixed
   first_allele_fixed = (history[-1][0] == N) if fixed else False

   allele_freq = history / N

   return {
       "fixed": fixed,
       "final_generations": final_t,
       "allele_counts_history": history,
       "allele_frequencies_history": allele_freq,
       "first_allele_fixed": first_allele_fixed
   }

def run_multiple_simulations_parallel(initial_K_counts: np.ndarray,
                                     Ff: List[Callable[[int], float]],
                                     num_simulations: int) -> List[Dict]:
    """
    runs multiple independent Wright–Fisher simulations in parallel at optimum efficiency

    params
     initial_K_counts : np.ndarray, initial allele counts
     Ff : List[Callable[[int], float]], list of fitness functions of each allele
     num_simulations : int

    output:e
     List[Dict]: list of dictionaries returned by run_single_simulation
    """
    if is_notebook():
        # Notebooks need the 'fork' start method to work with ProcessPoolExecutor
        try:
            multiprocessing.set_start_method("fork", force=True)
        except RuntimeError:
            pass  # Already set
        cfg_context = None
    else:
        # Scripts on Windows/macOS use 'spawn' by default, requiring a context or guard
        cfg_context = multiprocessing.get_context("spawn")

    with ProcessPoolExecutor(mp_context=cfg_context) as executor:
       futures = [
           executor.submit(run_single_simulation, initial_K_counts, Ff)
           for _ in range(num_simulations)
       ]
       results = [future.result() for future in futures]
    return results


# define a named fitness function (can't use lambdas)
def constant_fitness_1(t: int) -> float:
   return 1.0

def constant_fitness_2(t: int) -> float:
   return 2.0

def constant_fitness_3(t: int) -> float:
   return 3.0

def constant_fitness_4(t: int) -> float:
   return 4.0

def constant_fitness_5(t: int) -> float:
   return 5.0

def constant_fitness_6(t: int) -> float:
   return 6.0

def constant_fitness_7(t: int) -> float:
   return 7.0

def constant_fitness_8(t: int) -> float:
   return 8.0

def constant_fitness_9(t: int) -> float:
   return 9.0

def constant_fitness_10(t: int) -> float:
   return 10.0

def constant_fitness_11(t: int) -> float:
   return 11.0

def constant_fitness_12(t: int) -> float:
   return 12.0

def Psi_s(s: float) -> float:
    res = 1 + lambertw(-(1 + s) * np.exp(-(1 + s))). real / (s + 1)
    # bug in lambertw(-пр.exp(-1)), k=-1) == -1
    if isinstance(res, np.ndarray):
        res[np.isnan(res)] = 0.
    else:
        if np.isnan(res):
         res = 0.
    return res

"""
import sympy as sp
Psi, s = sp.symbols('Psi s', real=True)
sp.solve(sp.Eq(1 - Psi, sp.exp(-(1+s) * Psi)), Psi)
[s + sp.LambertW((-s - 1)*sp.exp(-s - 1)) + 1)/(s + 1)]
"""

def run_everything(INITIAL_ALLELE_COUNTS: np.ndarray, FITNESS_FUNCTIONS: List[Callable[[int], float]], NUM_RUNS: int) -> Dict:
    results = {}
    results['INITIAL_ALLELE_COUNTS'] = str(INITIAL_ALLELE_COUNTS)
    results['FITNESS_FUNCTIONS'] = [f.__name__ for f in FITNESS_FUNCTIONS]

    fitness1 = FITNESS_FUNCTIONS[0](1)
    fitness2 = FITNESS_FUNCTIONS[1](1)
    size1 = INITIAL_ALLELE_COUNTS[0]
    size2 = INITIAL_ALLELE_COUNTS[1]
    N = size1 + size2
    results['N'] = N

    ### PROBABILITY OF FIXATION

    # NEUTRAL ALLELE, any population
    neutral_allele_expected_probability_of_first_allele_fixation = INITIAL_ALLELE_COUNTS[0] / (INITIAL_ALLELE_COUNTS[0] + INITIAL_ALLELE_COUNTS[1])
    results['neutral_allele_expected_probability_of_first_allele_fixation'] = neutral_allele_expected_probability_of_first_allele_fixation

    # BENEFICIAL ALLELE

    selection_coefficient = (fitness1 - fitness2)/fitness2
    s = selection_coefficient # not efficient i know but it's to help me keep track, i'll fix later
    results['s'] = s

    # Beneficial allele, large population, one allele introduced
    beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_one_allele = Psi_s(s)
    results['beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_one_allele'] = beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_one_allele

    # Beneficial allele, large population, weak selection
    beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_weak_selection = 2*s
    results['beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_weak_selection'] = beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_weak_selection

    # Beneficial allele, weak selection
    denominator_weak_selection = -np.expm1(-4*N*s)

    if denominator_weak_selection != 0:
        beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection = -np.expm1(-2*s) / denominator_weak_selection
    else:
        beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection = np.nan # Handle division by zero
    results['beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection'] = beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection

    # ARBITRARY everything
    avg_fitness = ((size1 * fitness1) + (size2 * fitness2)) / N
    absolute_fitness = fitness1 / avg_fitness
    W = absolute_fitness
    r = np.log(W)

    expected_probability_of_first_allele_fixation_arbitrary_parameters = np.nan
    if r != 0: # Check to prevent division by zero in (1/r)
        term1_base = 1/r
        numerator = (1 - term1_base**INITIAL_ALLELE_COUNTS[0])
        denominator = (1 - term1_base**N)
        if denominator != 0: # Check to prevent division by zero
            expected_probability_of_first_allele_fixation_arbitrary_parameters = numerator / denominator
    results['expected_probability_of_first_allele_fixation_arbitrary_parameters'] = expected_probability_of_first_allele_fixation_arbitrary_parameters


    ### TIME TO FIXATION (Predictions)

    c = 1
    time_to_fixation_pred1 = np.nan
    if s != 0 and (c * INITIAL_ALLELE_COUNTS.sum() - 1) > 0: # Avoid division by zero and log of non-positive
        time_to_fixation_pred1 = 2 * np.log(c * INITIAL_ALLELE_COUNTS.sum() - 1) / s
    results['predicted_time_to_fixation_pred1'] = time_to_fixation_pred1


    # N is already defined
    p = size1 / N
    q = size2 / N

    mean_fitness = p * fitness1 + q * fitness2

    Vw = 0.0
    if mean_fitness != 0:
        Vw = p * q * (fitness1 - fitness2)**2 / mean_fitness**2
    else:
        Vw = np.nan # Handle if mean_fitness is 0

    Ne = N / (1 + Vw) # Ne will be inf if 1+Vw is 0. Handle if necessary.

    time_to_fixation_pred2 = np.nan
    if s != 0 and (c * Ne - 1) > 0 and np.isfinite(Ne): # Avoid division by zero, log of non-positive, and non-finite Ne
        time_to_fixation_pred2 = 2 * np.log(c * Ne - 1) / s
    results['predicted_time_to_fixation_pred2'] = time_to_fixation_pred2


    results['chosen_predicted_avg_fixation_time'] = np.nan
    if INITIAL_ALLELE_COUNTS[0] == 1 and s > 0 and np.isfinite(Ne):
      if s < 1 / Ne:
            results['chosen_predicted_avg_fixation_time'] = time_to_fixation_pred2
      else:
            results['chosen_predicted_avg_fixation_time'] = time_to_fixation_pred1


    ### Now run the simulations
    sim_data = run_multiple_simulations_parallel(initial_K_counts=INITIAL_ALLELE_COUNTS,
                                        Ff=FITNESS_FUNCTIONS,
                                        num_simulations=NUM_RUNS)

    fixed_generations = [run['final_generations'] for run in sim_data if run['fixed']]
    first_allele_fixations = [run for run in sim_data if run['first_allele_fixed']]

    simulated_average_fixation_time = np.nan
    if fixed_generations:
        simulated_average_fixation_time = np.mean(fixed_generations)
    results['simulated_avg_fixation_time'] = simulated_average_fixation_time

    first_allele_fixation_rate = len(first_allele_fixations) / NUM_RUNS
    results['first_allele_fixation_rate'] = first_allele_fixation_rate

    current_difference = abs(first_allele_fixation_rate - neutral_allele_expected_probability_of_first_allele_fixation)

    results["chosen_predicted_probability_of_first_allele_fixation"] = neutral_allele_expected_probability_of_first_allele_fixation

    # there must be a more efficient way to do the next few lines but idk what yet
    if current_difference > abs(first_allele_fixation_rate - beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_one_allele):
        results["chosen_predicted_probability_of_first_allele_fixation"] = beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_one_allele
        current_difference = abs(first_allele_fixation_rate - beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_one_allele)

    if current_difference > abs(first_allele_fixation_rate - beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_weak_selection):
        results["chosen_predicted_probability_of_first_allele_fixation"] = beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_weak_selection
        current_difference = abs(first_allele_fixation_rate - beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_weak_selection)

    if current_difference > abs(first_allele_fixation_rate - beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection):
        results["chosen_predicted_probability_of_first_allele_fixation"] = beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection
        current_difference = abs(first_allele_fixation_rate - beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection)

    if current_difference > abs(first_allele_fixation_rate - expected_probability_of_first_allele_fixation_arbitrary_parameters):
        results["chosen_predicted_probability_of_first_allele_fixation"] = expected_probability_of_first_allele_fixation_arbitrary_parameters
        current_difference = abs(first_allele_fixation_rate - expected_probability_of_first_allele_fixation_arbitrary_parameters)


    return results

def run_all_experiments(initial_allele_counts_list: List[np.ndarray],
                        fitness_function_pairs: List[List[Callable[[int], float]]],
                        num_runs: int) -> pd.DataFrame:
    all_results = []
    for initial_counts in initial_allele_counts_list:
        for ff_pair in fitness_function_pairs:
            result = run_everything(initial_counts, ff_pair, num_runs)
            all_results.append(result)
    return pd.DataFrame(all_results)


def main_runner():
    # Define the ranges for allele counts and fitness functions
    initial_allele_counts = [
        np.array([1, 20]), np.array([1, 1000]),
        np.array([2, 20]), np.array([3, 1000]),
        np.array([4, 20]), np.array([5, 1000]),
        np.array([6, 20]), np.array([7, 1000]),
        np.array([8, 20]), np.array([90, 1000]),
        np.array([10, 20]), np.array([700, 1000]),
        np.array([19, 20]), np.array([100, 1000]),
        np.array([11, 20]), np.array([9999, 1000]),
        np.array([3, 20]), np.array([400, 1000]),
        np.array([3, 20]), np.array([90, 1000]),
        np.array([12, 20]), np.array([4, 1000]),
        np.array([3, 20]), np.array([7, 1000]),
        np.array([2, 20]), np.array([900, 1000]),
        np.array([1, 20]), np.array([1, 1000]),
        np.array([2, 20]), np.array([3, 1000]),
        np.array([4, 20]), np.array([5, 1000]),
        np.array([6, 20]), np.array([7, 1000]),
        np.array([8, 20]), np.array([90, 1000]),
        np.array([10, 20]), np.array([700, 1000]),
        np.array([19, 20]), np.array([100, 1000]),
        np.array([11, 20]), np.array([9999, 1000]),
        np.array([3, 20]), np.array([400, 1000]),
        np.array([3, 20]), np.array([90, 1000]),
        np.array([12, 20]), np.array([4, 1000]),
        np.array([3, 20]), np.array([7, 1000]),
        np.array([2, 20]), np.array([900, 1000])
    ]

    fitness_function_pairs = [
        [constant_fitness_1, constant_fitness_1],
        [constant_fitness_2, constant_fitness_1],
        [constant_fitness_3, constant_fitness_1],
        [constant_fitness_4, constant_fitness_1],
        [constant_fitness_5, constant_fitness_1],
        [constant_fitness_6, constant_fitness_1],
        [constant_fitness_7, constant_fitness_1],
        [constant_fitness_8, constant_fitness_1],
        [constant_fitness_9, constant_fitness_1],
        [constant_fitness_10, constant_fitness_1],
        [constant_fitness_11, constant_fitness_1],
        [constant_fitness_12, constant_fitness_1],
        [constant_fitness_2, constant_fitness_3],
        [constant_fitness_1, constant_fitness_2],
        [constant_fitness_1, constant_fitness_3],
        [constant_fitness_1, constant_fitness_4],
        [constant_fitness_1, constant_fitness_5],
        [constant_fitness_1, constant_fitness_6],
        [constant_fitness_1, constant_fitness_7],
        [constant_fitness_1, constant_fitness_8],
        [constant_fitness_1, constant_fitness_9],
        [constant_fitness_1, constant_fitness_10],
        [constant_fitness_1, constant_fitness_11],
        [constant_fitness_1, constant_fitness_12]
    ]


    NUM_RUNS = 4000




    my_results = run_all_experiments(initial_allele_counts,
                                                      fitness_function_pairs,
                                                      NUM_RUNS)
   # Convert list of dictionaries to a DataFrame
    df = pd.DataFrame(my_results)

# Put columns in the order you want
    column_order = [
    "INITIAL_ALLELE_COUNTS",
    "FITNESS_FUNCTIONS",
    "N",
    "s",
    "first_allele_fixation_rate",
    "chosen_predicted_probability_of_first_allele_fixation",
    "neutral_allele_first_allele_fixation_rate",
    "beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_one_allele",
    "beneficial_allele_large_pop_expected_probability_of_first_allele_fixation_weak_selection",
    "beneficial_allele_expected_probability_of_first_allele_fixation_weak_selection",
    "expected_probability_of_first_allele_fixation_arbitrary_parameters",
    "simulated_avg_fixation_time",
    "chosen_predicted_avg_fixation_time",
    "predicted_time_to_fixation_pred1",
    "predicted_time_to_fixation_pred2",
    ]

# Keep only columns that actually exist
    column_order = [c for c in column_order if c in df.columns]
    df = df[column_order]

# Display nicely (Jupyter Notebook)
    styled = (
        df.style
          .set_caption("Simulation Results")
          .format(precision=6)
          .set_properties(**{
          "text-align": "center",
          "white-space": "normal"
          })
          .set_table_styles([
              {"selector": "th",
               "props": [
                   ("background-color", "#2E4057"),
                   ("color", "white"),
                   ("font-weight", "bold"),
                   ("text-align", "center"),
                   ("padding", "8px")
               ]},
              {"selector": "td",
               "props": [
                   ("padding", "6px"),
                   ("border", "1px solid #ddd")
               ]},
              {"selector": "caption",
               "props": [
                   ("font-size", "16px"),
                   ("font-weight", "bold"),
                   ("margin-bottom", "10px")
               ]},
          ])
          .background_gradient(cmap="Blues", subset=df.select_dtypes("number").columns)
    )

    styled

    from rich.console import Console
    from rich.table import Table

    table = Table(title="Simulation Results")

    for col in df.columns:
        table.add_column(str(col), justify="center", style="cyan")

    for _, row in df.iterrows():
        table.add_row(*[f"{x:.6g}" if isinstance(x, float) else str(x) for x in row])

    Console().print(table)

    df.to_html("results.html", index=False)


    # Convert to DataFrame
    df = pd.DataFrame(my_results)

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    # ............................................
    # Fixation probability
    # .............................................

    prob_df = df[
        [
            "first_allele_fixation_rate",
            "chosen_predicted_probability_of_first_allele_fixation",
        ]
    ].dropna()

    obs_prob = prob_df["first_allele_fixation_rate"]
    pred_prob = prob_df["chosen_predicted_probability_of_first_allele_fixation"]

    r2_prob = r2_score(obs_prob, pred_prob)
    mae_prob = mean_absolute_error(obs_prob, pred_prob)

    lims = [
        min(obs_prob.min(), pred_prob.min()),
        max(obs_prob.max(), pred_prob.max()),
    ]

    axes[0].scatter(obs_prob, pred_prob, s=60)
    axes[0].plot(lims, lims, "r--", label="Perfect prediction")
    axes[0].set_xlim(lims)
    axes[0].set_ylim(lims)
    axes[0].set_xlabel("Observed fixation probability")
    axes[0].set_ylabel("Predicted fixation probability")
    axes[0].set_title("Fixation Probability")
    axes[0].grid(alpha=0.3)

    axes[0].text(
        0.05,
        0.95,
        f"$R^2$ = {r2_prob:.3f}\nMAE = {mae_prob:.4f}",
        transform=axes[0].transAxes,
        verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.8),
    )

    # ............................................
    # Fixation time
    # ........................................

    time_df = df[
        [
            "simulated_avg_fixation_time",
            "chosen_predicted_avg_fixation_time",
        ]
    ].dropna()

    obs_time = time_df["simulated_avg_fixation_time"]
    pred_time = time_df["chosen_predicted_avg_fixation_time"]

    r2_time = r2_score(obs_time, pred_time)
    mae_time = mean_absolute_error(obs_time, pred_time)

    lims = [
        min(obs_time.min(), pred_time.min()),
        max(obs_time.max(), pred_time.max()),
    ]

    axes[1].scatter(obs_time, pred_time, s=60, color="darkorange")
    axes[1].plot(lims, lims, "r--", label="Perfect prediction")
    axes[1].set_xlim(lims)
    axes[1].set_ylim(lims)
    axes[1].set_xlabel("Observed fixation time")
    axes[1].set_ylabel("Predicted fixation time")
    axes[1].set_title("Fixation Time")
    axes[1].grid(alpha=0.3)

    axes[1].text(
        0.05,
        0.95,
        f"$R^2$ = {r2_time:.3f}\nMAE = {mae_time:.3f}",
        transform=axes[1].transAxes,
        verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.8),
    )

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main_runner()




/var/folders/vs/hjyn9v717lgfw260xy21bmh40000gn/T/ipykernel_7081/948794590.py:241: RuntimeWarning: overflow encountered in expm1
  denominator_weak_selection = -np.expm1(-4*N*s)
/var/folders/vs/hjyn9v717lgfw260xy21bmh40000gn/T/ipykernel_7081/948794590.py:259: RuntimeWarning: overflow encountered in scalar power
  denominator = (1 - term1_base**N)
/var/folders/vs/hjyn9v717lgfw260xy21bmh40000gn/T/ipykernel_7081/948794590.py:258: RuntimeWarning: overflow encountered in scalar power
  numerator = (1 - term1_base**INITIAL_ALLELE_COUNTS[0])
/var/folders/vs/hjyn9v717lgfw260xy21bmh40000gn/T/ipykernel_7081/948794590.py:261: RuntimeWarning: invalid value encountered in scalar divide
  expected_probability_of_first_allele_fixation_arbitrary_parameters = numerator / denominator
